# 06. 源码阅读与工程实践

这一章面向想读 SimPy 源码、改示例、或把 SimPy 用到工程仿真的读者。重点是把官方文档中的概念映射到源码结构和实际建模方法。

## 官方文档和源码的对应关系

| 官方文档主题 | 主要源码位置 | 阅读重点 |
| --- | --- | --- |
| Environments | `simpy/core.py` | 事件队列、`run()`、`step()`、`schedule()` |
| Events | `simpy/events.py` | `Event`、`Timeout`、`Process`、`Condition` |
| Process Interaction | `simpy/events.py` | 进程恢复、中断、进程作为事件 |
| Shared Resources | `simpy/resources/resource.py` | `Resource`、请求队列、释放逻辑 |
| Containers | `simpy/resources/container.py` | 连续容量、`put()`、`get()` |
| Stores | `simpy/resources/store.py` | 对象队列、优先级、过滤 |
| Real-time simulations | `simpy/rt.py` | 仿真时间和真实时间同步 |
| Utilities | `simpy/util.py` | 延迟启动、事件订阅 |

建议阅读顺序：

1. `simpy/events.py` 中的 `Event`。
2. `simpy/core.py` 中的 `Environment.schedule()`、`step()`、`run()`。
3. `simpy/events.py` 中的 `Timeout` 和 `Process`。
4. `simpy/events.py` 中的 `Condition`、`AnyOf`、`AllOf`。
5. `simpy/resources/base.py` 和具体资源实现。

## Environment 的核心职责

`Environment` 是调度中心。它通常维护：

- 当前仿真时间。
- 一个按事件时间排序的优先队列。
- 事件编号，用于同一时间点排序。
- 创建事件和进程的快捷方法。

从行为上看，核心循环类似：

```python
while event_queue:
    time, priority, event_id, event = heapq.heappop(event_queue)
    now = time
    process_event_callbacks(event)
```

真实源码会处理更多细节，例如失败事件、空队列、停止条件和回调清理，但主干就是“按时间顺序处理事件”。

## Event 的核心字段

阅读事件源码时，重点关注这些信息：

| 字段或概念 | 作用 |
| --- | --- |
| `env` | 事件属于哪个环境 |
| `callbacks` | 事件被处理时执行的回调 |
| `_value` | 事件成功值或失败异常 |
| 触发状态 | 事件是否已有结果 |
| 处理状态 | 事件是否已经被环境处理 |

进程等待事件时，本质上是把“恢复这个进程”的逻辑加入事件回调中。

## Process 如何恢复

`Process` 包装一个 Python 生成器。它的工作方式可以理解为：

1. 启动进程。
2. 调用生成器的 `send()` 或 `throw()`。
3. 生成器运行到下一个 `yield`。
4. 得到一个事件。
5. 给这个事件注册回调：事件完成后继续恢复进程。
6. 如果生成器结束，进程事件成功。
7. 如果生成器抛异常，进程事件失败。

这解释了为什么 SimPy 代码既像同步代码，又能表达并发流程。

## Condition 如何组合事件

`AnyOf` 和 `AllOf` 都是条件事件。

- `AnyOf`：任一子事件成功，条件事件就可以成功。
- `AllOf`：所有子事件成功，条件事件才成功。

条件事件会监听多个子事件的完成情况，然后把完成事件和值收集起来。

运算符只是语法糖：

```python
event_a | event_b  # AnyOf
event_a & event_b  # AllOf
```

阅读源码时，可以把条件事件理解为“带计数和结果收集的事件回调管理器”。

## 资源的源码结构

资源通常建立在通用 `put/get` 机制上。

普通 `Resource` 的 `request()` 和 `release()` 可以理解为特殊的 `put/get`：

- 请求资源：如果容量未满，请求事件立即成功；否则进入等待队列。
- 释放资源：释放一个占用名额，然后检查等待队列。

`Container`：

- `put(amount)` 等待有足够剩余容量。
- `get(amount)` 等待有足够当前库存。

`Store`：

- `put(item)` 等待有空位。
- `get()` 等待有对象。

资源类的共同点是：操作不一定立即完成，而是返回事件。

## 建模边界

SimPy 提供的是仿真机制，不是完整业务框架。它不负责：

- 自动生成输入数据。
- 自动验证模型假设。
- 自动输出统计报表。
- 自动绘图。
- 自动并行多次实验。
- 自动优化参数。

工程中通常需要在 SimPy 外围补充：

- 参数配置层。
- 随机分布层。
- 结果收集层。
- 批量实验脚本。
- 可视化脚本。
- 测试和校验数据。

## 工程建模推荐结构

一个清晰的仿真工程可以拆成：

```text
simulation/
  config.py        # 参数和分布
  model.py         # SimPy 进程和资源定义
  metrics.py       # 指标记录和统计
  experiments.py   # 多组参数实验
  plots.py         # 绘图
  tests/           # 小规模确定性测试
```

小项目可以放在一个文件里；但一旦有多个资源、多类请求和多组实验，就应该拆分。

## 指标设计

常见指标及记录位置：

| 指标 | 记录时刻 |
| --- | --- |
| 到达时间 | 进程创建或请求进入系统时 |
| 排队等待 | 获得资源时 |
| 服务时间 | 服务开始前抽样，服务结束后确认 |
| 完成时间 | 流程结束时 |
| 系统逗留时间 | 完成时间减到达时间 |
| 队列长度 | 资源请求、释放、固定采样或事件打点时 |
| 利用率 | 资源占用数量变化时 |
| 超时数量 | `AnyOf` 中超时分支发生时 |
| 中断数量 | 捕获 `simpy.Interrupt` 时 |

建议把原始事件日志保存下来，再计算汇总指标。这样更容易复查模型是否正确。

## 绘图建议

SimPy 常见绘图有三类：

### 1. 时间序列图

适合：

- 队列长度。
- 资源占用数。
- 库存量。
- 系统中请求数量。

推荐阶梯图，因为状态通常只在事件点变化。

```python
df.plot(x="time", y="queue", drawstyle="steps-post")
```

### 2. 分布图

适合：

- 等待时间分布。
- 响应时间分布。
- 服务时间分布。

可以用直方图、箱线图、经验分布函数。

### 3. 多次实验汇总图

适合比较不同参数：

- 不同资源数量下的平均等待时间。
- 不同到达率下的超时比例。
- 不同调度策略下的完成率。

不要只用单次随机结果做结论。至少应重复多次，报告均值、标准差或置信区间。

## 测试 SimPy 模型

仿真模型也需要测试。推荐从确定性小例子开始：

```python
def test_single_server_wait_time():
    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    waits = []

    env.process(customer(env, "A", server, service_time=3, waits=waits))
    env.process(customer(env, "B", server, service_time=2, waits=waits))
    env.run()

    assert waits == [0, 3]
```

测试重点：

- 资源排队顺序是否正确。
- 超时分支是否正确。
- 中断后资源是否释放。
- 指标时间点是否正确。
- 随机种子固定时结果是否可复现。

## 常见源码阅读问题

### 为什么事件有时候触发了，但进程还没继续？

触发和处理不是同一件事。事件触发表示它已经有结果；进程恢复发生在环境处理该事件的回调时。

### 为什么 `env.timeout(0)` 也会让出控制权？

它创建的是当前仿真时间的事件。时间不前进，但调度器仍会在事件队列中处理它，所以可以用于让其他同一时间点的进程获得执行机会。

### 为什么同一时间点输出顺序和我想的不一样？

同一时间点事件仍有内部优先级和创建顺序。业务上需要严格优先级时，应使用显式优先级资源或事件，而不是依赖输出顺序。

### 为什么仿真没有结束？

常见原因：

- 存在无限循环进程，并且没有 `until`。
- 某些后台监控进程一直 `yield env.timeout(...)`。
- 资源等待永远无法满足。

解决方法：

- 使用 `env.run(until=...)`。
- 给后台进程设置结束条件。
- 检查队列、库存和资源容量是否可能死锁。

## 和 faas-sim 类系统的映射

如果在函数计算或服务器仿真中使用 SimPy，可以按以下方式建模：

| 系统概念 | SimPy 表达 |
| --- | --- |
| 函数请求 | 进程或 `Store` 中的任务对象 |
| 请求到达流 | 到达生成器进程 |
| 函数实例 | `Store` 中的实例对象，或 `Resource` 容量 |
| CPU 并发槽 | `Resource` |
| 内存容量 | `Container` |
| 冷启动 | `yield env.timeout(cold_start)` |
| 执行时间 | `yield env.timeout(runtime)` |
| 调度器 | 长期运行的进程 |
| 请求队列 | `Store` 或 `PriorityStore` |
| 异构节点匹配 | `FilterStore` |
| 优先级任务 | `PriorityStore` 或 `PriorityResource` |
| 抢占执行 | `PreemptiveResource` 和中断 |
| 超时 | `AnyOf(request_done, timeout)` |

这类系统通常不要把所有逻辑塞进一个进程。更清晰的拆法是：

- 请求生命周期进程。
- 调度器进程。
- 节点或实例管理进程。
- 监控进程。
- 结果收集器。

## 代码质量建议

### 保持时间单位统一

明确所有时间是秒、毫秒还是分钟。不要让到达间隔是秒而服务时间是毫秒。

### 随机数集中管理

复杂模型中建议把随机分布放入配置或函数：

```python
def sample_service_time(rng):
    return rng.expovariate(1 / 3)
```

这样便于复现实验和替换分布。

### 不要在进程中隐藏太多全局状态

优先把资源、配置、指标作为参数传入进程函数。这样更容易测试。

### 统计和业务逻辑分离

可以先在业务关键点记录事件日志：

```python
metrics.append(("start", env.now, request_id))
metrics.append(("finish", env.now, request_id))
```

再单独写函数计算均值、分位数和利用率。

### 小规模手算校验

在引入随机分布前，先用固定到达时间和固定服务时间跑小例子。手算结果和仿真结果一致后，再放开随机性。

## 何时不要用 SimPy

以下场景不建议强行使用 SimPy：

- 主要问题是数值微分方程求解。
- 需要真实并行执行大量计算任务，而不是仿真任务。
- 需要三维物理引擎或图形渲染。
- 只需要简单公式计算，不需要事件调度。
- 系统状态连续变化且不能离散化为事件。

## 小结

源码阅读时，把 SimPy 看成两层：

- 底层：事件队列、事件触发、进程恢复。
- 上层：资源、容器、存储和业务进程。

工程实践中，把 SimPy 看成一个仿真内核：

- 它负责时间和事件。
- 你负责模型结构、输入假设、统计指标和结果解释。

这也是官方文档的核心思路：先理解事件和进程，再用资源模型表达系统约束，最后通过案例和指标把模型落到真实问题上。
